In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split

In [2]:
# !wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
# !mv car_fuel_efficiency_2026.csv ../data/

In [3]:
df = pd.read_csv("../data/car_fuel_efficiency_2026.csv")

In [4]:
# Fill missing values with Zero
df.horsepower = df.horsepower.fillna(0)
df.acceleration = df.acceleration.fillna(0)

In [5]:
# Split the dataframe
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=11)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=11)

# Reset index
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Prepare target variable
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Drop the target variable from the training dataset
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [6]:
dv = DictVectorizer(sparse=True)

# transform dataframes to dicts
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

# X_train, X_val, X_test
X_train = dv.fit_transform(train_dicts)
X_val =  dv.transform(val_dicts)
X_test =  dv.transform(test_dicts)

In [12]:
feature_names = list(dv.get_feature_names_out())

In [8]:
y_train

array([37.3, 32.8, 30.2, ..., 30.8, 31.7, 31.1], shape=(6000,))

In [17]:
'''
    Question 1
    Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
'''
from sklearn.tree import DecisionTreeRegressor, export_text
max_depth = 1

dt = DecisionTreeRegressor(max_depth=max_depth)
dt.fit(X_train, y_train)

print(export_text(decision_tree=dt, feature_names=feature_names))

# Answer: model_year

|--- model_year <= 1995.50
|   |--- value: [28.47]
|--- model_year >  1995.50
|   |--- value: [31.10]



In [23]:
'''
Question 2
Train a random forest regressor with these parameters:

n_estimators=10
random_state=1
n_jobs=-1 (optional - to make training faster)
What's the RMSE of this model on the validation data?
'''
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)
rmse

# Answer: 1.803742678432819

1.803742678432819

In [27]:
'''
Question 3
Now let's experiment with the n_estimators parameter

Try these values of this parameter: [10, 50, 100, 150].
Set random_state to 1.
Evaluate the model on the validation dataset.
Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.
'''

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

for est in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=est, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    print(f"rmse_{est}={round(rmse, 3)}")

# Answer: 150

rmse_10=1.804
rmse_50=1.748
rmse_100=1.738
rmse_150=1.733


In [33]:
'''
Let's select the best max_depth:

Try different values of max_depth: [10, 15, 20, 25]
For each of these values,
try different values of n_estimators from [10, 50, 100, 150]
calculate the mean RMSE
Fix the random seed: random_state=1
What's the best max_depth, using the mean RMSE?
'''

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

for depth in [10, 15, 20, 25]:
    # print(f"depth_{depth}")
    rmses = []
    for est in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=est, max_depth= depth, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_val)
        
        rmse = root_mean_squared_error(y_val, y_pred)
        rmses.append(rmse)
        # print(f"rmse for n_estimators={est} is {round(rmse, 3)}")

    avg = np.mean(rmses)
    print(f"depth={depth}, mean_rmse={round(avg, 3)}")
    print()

# Answer: 10

depth=10, mean_rmse=1.725

depth=15, mean_rmse=1.754

depth=20, mean_rmse=1.757

depth=25, mean_rmse=1.757



In [67]:
'''
Question 5
We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

For this homework question, we'll find the most important feature:

Train the model with these parameters:
n_estimators=10,
max_depth=20,
random_state=1,
n_jobs=-1 (optional)
Get the feature importance information from this model
What's the most important feature (among these 4)?
'''

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = rf.feature_importances_

prob = {}
for i in range(len(feature_names)):
    key = feature_names[i]
    if key in ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]:
        value = float(round(importances[i], 4))
        prob[key] = value
print(prob)

# Answer: vehicle_weight

{'acceleration': 0.0555, 'engine_displacement': 0.063, 'horsepower': 0.0762, 'vehicle_weight': 0.2036}


In [55]:
'''
Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

Install XGBoost
Create DMatrix for train and validation
Create a watchlist
Train a model with these parameters for 100 rounds:
    xgb_params = {
        'eta': 0.3, 
        'max_depth': 6,
        'min_child_weight': 1,
        
        'objective': 'reg:squarederror',
        'nthread': 8,
        
        'seed': 1,
        'verbosity': 1,
    }
Now change eta from 0.3 to 0.1.

Which eta leads to the best RMSE score on the validation dataset?
'''

import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
rounds = 100


# Wrapping the data into DMatrix. This is because XGBoost does not work with numpy arrays
# It works with a special data structure called "DMatrix" which is optimised for training xgboost models
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

res = {}

for eta in [0.3, 0.1]:
    evals_result = {}
    # watchlist = [(dtrain, 'train'), (dval, 'val')]
    
    model = xgb.train(params=xgb_params,
                      dtrain=dtrain,
                      num_boost_round=rounds,
                      # evals=watchlist,
                      verbose_eval=50,
                      evals_result=evals_result)
    y_pred = model.predict(dval)
    key = f"eta={eta}"
    rmse = root_mean_squared_error(y_val, y_pred)
    res[key] = rmse
    
res

# Answer: Both give equal value

{'eta=0.3': 1.805296841758663, 'eta=0.1': 1.805296841758663}